# Descarga de Datos

En esta libreta se gestionara la descarga de los datos para nuestro analisis.

## Introducción.
Como primer paso haremos la selección del conjunto de datos a analizar. En nuestro proyecto nos centraremos en la parte Agrícola. Sin embargo, es posible acceder a la información Pecuaria.

Para la descarga de datos usaremos las librerias BeautifulSoup and Requests.

In [13]:
import requests
from bs4 import BeautifulSoup
import yarl
import re
import pandas as pd
from pathlib import Path
import os

DATA_PATH = Path("..") / "data" / "external"
print(DATA_PATH)

../data/external


In [14]:
# Pagina principal de los Datos.
MAIN_URL = yarl.URL("https://nube.agricultura.gob.mx")
DGSIAP_URL = MAIN_URL / "datosAbiertos/"
DATA_URL = MAIN_URL / "index.php"


print("Bienvenido al script de descarga de datos abiertos del DGSIAP.")
info = {
    "Agricola": "Agricola.php",
    "Pecuaria": "Pecuario.php",
}

TEST = 1

if not TEST:
    value = input(f"Ingrese la categoria de los datos a consultar. {" o ".join(info.keys())}: ")

    try:
        DGSIAP = DGSIAP_URL / info[value]
    except KeyError:
        print("Categoria no valida. Por favor, ingrese una categoria valida.")
        exit(1)
else:
    DGSIAP = DGSIAP_URL / info["Agricola"]


print(DGSIAP)
response = requests.get(DGSIAP)
if response.status_code != 200:
    print(f"Error al acceder a la pagina: {response.status_code}")
    exit(1)

dgsiap = BeautifulSoup(response.content, "lxml")

Bienvenido al script de descarga de datos abiertos del DGSIAP.
https://nube.agricultura.gob.mx/datosAbiertos/Agricola.php


Notemos que una vez definida la categoria.
* Agricola
* Pecuaria.

BeatifulSoup nos permite explorar la pagina con HTML. Elemento que aprovecharemos para encontrar la información que requeriremos.

## Introducción a la pagina

In [15]:
content = dgsiap.find("main")

print(content)
sections = content.find_all("section", recursive=False)
tree = sections[0]


<main class="page" id="content" role="main" tabindex="-1">
<section class="content-box mb-0 py-0 bg-background-light">
<div class="container-fluid px-0">
<div class="top-buffer text-left">
<ol class="breadcrumb">
<li><a href="https://www.gob.mx/agricultura/" target="_blank"><i class="icon icon-home"></i></a></li>
<li><a href="https://www.gob.mx/agricultura/dgsiap" target="_top">Dirección General del Servicio de Información Agroalimentaria y Pesquera</a></li>
<li><a href="/datosAbiertos/index.php" target="_top">Datos abiertos</a></li>
<li class="active">Información agrícola</li>
</ol>
</div>
</div>
</section>
<section class="content-box mb-2 pt-0">
<div class="h3 font-weight-bold">
						Datos abiertos					</div>
<hr class="red"/>
<div class="row mb-3" style="margin-top:-20px;">
<div class="col-sm">
<div class="h4 font-weight-bold">
								Estadística de Producción Agrícola
							</div>
</div>
</div>
<div class="row mb-5">
<div class="col-sm text-justify header-content-title">
							

Vemos que la primera sección corresponde al arbol definido en la propia pagina. Notemos que de ahi podemos extraer información relevante.

In [16]:
for t in tree.find_all("li"):
    if t.text!="":
        print(t.text)

Dirección General del Servicio de Información Agroalimentaria y Pesquera
Datos abiertos
Información agrícola


## Descripcion

In [17]:
intro = sections[1]
raw_intro=list()
for t in intro.find_all("div", recursive=False):
    if t.text!="":
        raw_block=(t.text).split("\t")
        for raw in raw_block:
            if raw.strip() != "":
                raw_intro.append(raw[:-2])

print(raw_intro)

['Datos abiert', 'Estadística de Producción Agrícol', 'Los usuarios de la información podrán disponer de los datos anualizados de estadística básica agrícola a nivel nacional, estatal, por Distrito de Desarrollo Rural y municipal, la cual será de gran utilidad para el análisis del subsector y la toma de decisiones públicas y privadas que contribuyan al desarrollo rural sustentable del país']


## Ultima publicación

In [18]:
last = sections[2]
search_title_section=last.find("div")
last_data = last.find("div", {"class":"bento-card h-100"})
last_dict = last.find("div", {"class":"dict-section"})
ld_bs = last_dict.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
# print(last_dict)

### Diccionario de Datos

In [19]:
catalog, metadata = ld_bs[0], ld_bs[1]

# print(catalog)
b_title = catalog.find("h2").text
b_description=catalog.find("p").text
b_download = catalog.find("a")['href']
print(b_title, b_description, b_download,"\n")

print(DATA_URL)

DOWNLOAD_DATA=DATA_URL.update_query(view=b_download.split("view=")[-1])

file = requests.get(DOWNLOAD_DATA)
file_h = requests.head(DOWNLOAD_DATA)

def get_filename(url):
    url = requests.head(url)
    for u, v in url.headers.items():
        try:
            if v.index("filename"):
                name=v.split("filename=")[-1]
                name=name[1:-1]
                break
        except:
            pass
    return name

def download(url, path = DATA_PATH):
    filename = get_filename(url)
    linkdata = requests.get(url)
    filepath = path / filename
    print(filepath)
    with linkdata as lf:
        with open(filepath, "wb") as f:
            f.write(lf.content)
    

download(DOWNLOAD_DATA)


Diccionario de datos Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos. /index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7 

https://nube.agricultura.gob.mx/index.php
../data/external/Diccionario_agricola_2003_a_2025.xlsx


### Metadata

Para obtener los datos. Lo haremos de una forma similar. 

In [20]:
meta = metadata.find("div")
datetype = meta.find_all("div", recursive=False)

metafile = {
    "title": meta.find("h4").text,
    "state": datetype[0].find("span", {"class":"label"}).text,
    "lastdate": datetype[0].find("span", {"class":"value"}).text,
    "range": datetype[1].find("span", {"class":"label"}).text,
    "size": datetype[1].find("span", {"class":"value"}).text
}

print(metafile)



{'title': 'Metadatos Principales', 'state': 'Actualizado', 'lastdate': '12 agosto 2026', 'range': 'Cobertura 2003 - 2025', 'size': 'Nivel Municipal'}


### Conjunto de Datos

In [ ]:
u = last_data.find_all("div", recursive=False)
data_info = [s.text for s in u[0].find_all("span", recursive=False)]
data_link = u[1]
#print(data_info)Pecuaria

#print(data_link)
b_download = data_link.find("a")['href']
b_query = b_download.split("?")[-1].split("&")

print(b_query)
DOWNLOAD_DATA=DATA_URL.update_query(view=b_query[0].split("=")[-1],ANIO=b_query[1].split("=")[-1])

template = "https://nube.agricultura.gob.mx/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025"

download(DOWNLOAD_DATA)

## Descargar Datos Historicos

In [21]:
history = sections[3]
data_title = history.find("div", {"class":"mb-0 h5 font-weight-bold section-title"}).text
data_content = history.find("div", {"id":"dataAccordion"})
blocks = data_content.find_all("div", recursive=False)

data_blocks = {b.find("div", {"class":"d-flex align-items-center"}).text.strip(): 0 for b in blocks}
print(data_blocks)

b0 = blocks[1]
con_data = b0.find_all("a", {"class":"dl-btn"})
href_data = [yarl.URL(a['href']) for a in con_data]
data_querys = [hf.query for hf in href_data]


{'Municipal': 0, 'Nacional': 0, 'No seguimiento': 0}


## Diccionarios Datos Historicos

In [ ]:
d0 = b0.find("div", {"class":"dict-section"})

# print(d0)
ld_bs = d0.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
print(ld_bs[0])
print(ld_bs[1])

